this code is written heavily copied from andrej karphathy from the video and he was open soures code on github and video on yt plz consider it . dataset then reading and explaration data


In [31]:
#dataset
!curl https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt


  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0curl: (6) Could not resolve host: raw.githubusercontent.com


In [32]:
with open('shakespare.txt', 'r') as f:
    data = f.read()
    

In [33]:
print("length of dataset in characters: ", len(data))


length of dataset in characters:  1115394


In [34]:
print(data[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [35]:
chars = sorted(list(set(data)))
vocab_size = len(chars)
print('Characters:', ''.join(chars))
print('Vocabulary size:', vocab_size)

Characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocabulary size: 65


In [36]:
stoi  = {ch:i for i , ch in enumerate (chars)}
itos = { i:ch for i, ch in enumerate (chars)}
encode = lambda s:[stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("abcdefg"))
print(decode(encode("abcdefg")))

[39, 40, 41, 42, 43, 44, 45]
abcdefg


In [37]:
import torch 
lata = torch.tensor(encode(data), dtype=torch.long)
print(lata.shape, lata.dtype)
print(lata[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [38]:
n = int(0.9 * len(lata))
train_data = lata[:n]
val_data = lata[n:]

In [39]:
block_size = 8 
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [40]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t + 1]
    target = y[t]
    print (f"when the input is {context} and the target is {target}")

when the input is tensor([18]) and the target is 47
when the input is tensor([18, 47]) and the target is 56
when the input is tensor([18, 47, 56]) and the target is 57
when the input is tensor([18, 47, 56, 57]) and the target is 58
when the input is tensor([18, 47, 56, 57, 58]) and the target is 1
when the input is tensor([18, 47, 56, 57, 58,  1]) and the target is 15
when the input is tensor([18, 47, 56, 57, 58,  1, 15]) and the target is 47
when the input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) and the target is 58


In [41]:
torch.manual_seed(1337)


batch_size = 4
block_size = 8
def get_batch(split):
    lata = train_data if split == 'train' else val_data
    ix = torch.randint(len(lata)- block_size, (batch_size,))

    x = torch.stack([lata[i:i + block_size] for i in ix])
    y = torch.stack([lata[i + 1:i + 1 + block_size] for i in ix])
    return x, y


xb ,yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('\ntargets:')
print(yb.shape) 
print(yb)
print('----')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t + 1]
        target = yb[b, t]
        print(f"when input is {context.tolist()}the target : {target.item()}")
#wtf is xb and yb? : xb is the input batch and yb is the target batch
#where is the loss function? :



inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])

targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24]the target : 43
when input is [24, 43]the target : 58
when input is [24, 43, 58]the target : 5
when input is [24, 43, 58, 5]the target : 57
when input is [24, 43, 58, 5, 57]the target : 1
when input is [24, 43, 58, 5, 57, 1]the target : 46
when input is [24, 43, 58, 5, 57, 1, 46]the target : 43
when input is [24, 43, 58, 5, 57, 1, 46, 43]the target : 39
when input is [44]the target : 53
when input is [44, 53]the target : 56
when input is [44, 53, 56]the target : 1
when input is [44, 53, 56, 1]the target : 58
when input is [44, 53, 56, 1, 58]the target : 46
when input is [44, 5

In [42]:
print(xb)

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [51]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import torch
torch.manual_seed(1337)


class Bigramlanguagemodel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)
        def forward(self, idx, targets):

            logits = self.token_embedding_table(idx)
            return logits
    
m = Bigramlanguagemodel(vocab_size)
import torch.nn as nn
torch.manual_seed(1337)

class Bigramlanguagemodel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)

    def forward(self, idx, targets):
        logits = self.token_embedding_table(idx)
        loss = F.cross_entropy(logits, targets)
        B , T ,C = logits.shape
        return logits, loss
    def generate(self,idx,max_new_tokens):
        for _ in range(max_new_tokens):
            logits,loss = self(idx)
            logits = logits[:,-1,:]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs,num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
m = Bigramlanguagemodel(vocab_size)
out, loss = m(xb, yb)
print(out.shape)
print(loss)
#whats the issue here ? : the issue is that the forward method is not defined correctly, it should return logits and loss, but it only returns logits.
#help me : the issue is that the forward method is not defined correctly, it should return logits and loss, but it only returns logits. To fix this, we need to change the forward method to return both logits and loss. Here is the corrected code:
#what is the correct code ?:

RuntimeError: Expected target size [4, 65], got [4, 8]